In [ ]:
!pip install sentence-transformers faiss-cpu anthropic python-dotenv -q

In [ ]:
from dataclasses import dataclass

@dataclass
class Document:
    page_content: str
    metadata: dict

    def __repr__(self):
        preview = self.page_content[:80].replace('\n', ' ')
        keys = list(self.metadata.keys())
        return f"Document(preview='{preview}...', metadata_keys={keys})"

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.WARNING)

def load_documents(directory: str) -> list[Document]:
    path = Path(directory)

    if not path.exists():
        logging.warning(f"Directory not found: {directory}")
        return []

    documents = []

    for file in path.glob("*.txt"):
        text = file.read_text(encoding="utf-8", errors="ignore")
        doc = Document(
            page_content=text,
            metadata={
                "filename": file.name,
                "filepath": str(file),
                "source": str(file),
                "char_count": len(text)
            }
        )
        documents.append(doc)

    return documents

In [ ]:
docs = load_documents(".")
print(f"Loaded {len(docs)} documents\n")
for doc in docs:
    print(doc)
    print()

Loaded 0 documents



That was to handel TXT file now we will handel pdfs

In [ ]:
!pip install pypdf -q

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import logging

logging.basicConfig(level=logging.WARNING)

@dataclass
class Document:
    page_content: str
    metadata: dict

    def __repr__(self):
        preview = self.page_content[:80].replace('\n', ' ')
        keys = list(self.metadata.keys())
        return f"Document(preview='{preview}...', metadata_keys={keys})"


def load_txt(file: Path) -> str:
    return file.read_text(encoding="utf-8", errors="ignore")


def load_pdf(file: Path) -> str:
    from pypdf import PdfReader
    reader = PdfReader(str(file))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def load_documents(directory: str) -> list[Document]:
    path = Path(directory)

    if not path.exists():
        logging.warning(f"Directory not found: {directory}")
        return []

    documents = []

    for file in path.iterdir():
        if file.suffix == ".txt":
            text = load_txt(file)
        elif file.suffix == ".pdf":
            text = load_pdf(file)
        else:
            continue

        doc = Document(
            page_content=text,
            metadata={
                "filename": file.name,
                "filepath": str(file),
                "source": str(file),
                "char_count": len(text),
                "file_type": file.suffix
            }
        )
        documents.append(doc)

    return documents

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Replace 'BFL' with your actual folder name exactly as it appears in Drive
folder_path = '/content/drive/MyDrive/BFL'

# Confirm it exists and see what's inside
os.listdir(folder_path)

['AI & IP in Education_October 2025.pdf',
 'Alberta Colleges - 2025 Symposium - FM Presentation (final).pdf',
 'ACUTI Claims 101 Oct 2025.pdf',
 'AB Colleges 2025 Symposium - Edmonton October 2025 - Presentation.pdf']

In [ ]:
# Then load directly from your BFL folder
docs = load_documents('/content/drive/MyDrive/BFL')

print(f"Loaded {len(docs)} documents\n")
for doc in docs:
    print(doc)
    print()

Loaded 4 documents

Document(preview='BFL CANADA  Risk and Insurance Services Inc. | Financial Services Firm | bflcana...', metadata_keys=['filename', 'filepath', 'source', 'char_count', 'file_type'])

Document(preview='FM:  Your partner in propertyloss prevention and insuranceKen Lee & Marc-Andre C...', metadata_keys=['filename', 'filepath', 'source', 'char_count', 'file_type'])

Document(preview='1BFL CANADA Risk and Insurance Services Inc. | Financial Services Firm | bflcana...', metadata_keys=['filename', 'filepath', 'source', 'char_count', 'file_type'])

Document(preview='1BFL CANADA Risk and Insurance Services Inc. | Financial Services Firm | bflcana...', metadata_keys=['filename', 'filepath', 'source', 'char_count', 'file_type'])



# Chunking

In [ ]:
def chunk_documents(documents: list[Document],
                    chunk_size: int = 500,
                    overlap: int = 50) -> list[Document]:
    chunks = []

    for doc in documents:
        text = doc.page_content
        start = 0
        chunk_index = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            chunk = Document(
                page_content=chunk_text,
                metadata={
                    **doc.metadata,           # inherit all parent metadata
                    "chunk_index": chunk_index,
                    "chunk_start": start,
                    "chunk_end": end,
                }
            )
            chunks.append(chunk)
            start += chunk_size - overlap
            chunk_index += 1

    return chunks

In [ ]:
chunks = chunk_documents(docs, chunk_size=500, overlap=50)

print(f"Documents: {len(docs)}")
print(f"Chunks: {len(chunks)}")
print(f"Avg chunks per document: {len(chunks)/len(docs):.1f}\n")

# Inspect the first 3 chunks
for chunk in chunks[:3]:
    print(chunk)
    print(f"Text: {chunk.page_content[:200]}")
    print()

Documents: 4
Chunks: 166
Avg chunks per document: 41.5

Document(preview='BFL CANADA  Risk and Insurance Services Inc. | Financial Services Firm | bflcana...', metadata_keys=['filename', 'filepath', 'source', 'char_count', 'file_type', 'chunk_index', 'chunk_start', 'chunk_end'])
Text: BFL CANADA  Risk and Insurance Services Inc. | Financial Services Firm | bflcanada.ca 
BFL CANADA  Risk and Insurance Services Inc. | Financial Services Firm | bflcanada.ca 
October 23, 2025| 13:30 MT

Document(preview='Firm | bflcanada.ca  Discussion Flow  1. Introduction to AI & IP  2. Context & R...', metadata_keys=['filename', 'filepath', 'source', 'char_count', 'file_type', 'chunk_index', 'chunk_start', 'chunk_end'])
Text: Firm | bflcanada.ca 
Discussion Flow 
1. Introduction to AI & IP 
2. Context & Relevance: Education & 
Academia 
3. Key Concepts : AI, IP and the Academic 
Ecosystem 
4. Emerging Risks and Opportuniti

Document(preview='inancial Services Firm | bflcanada.ca  Artificial Intelligence

# Embedding

In [ ]:
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model once — expensive to reload repeatedly
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def embed_chunks(chunks: list[Document]) -> tuple[list[Document], list]:
    texts = [chunk.page_content for chunk in chunks]

    print(f"Embedding {len(texts)} chunks...")
    vectors = embedder.encode(texts, show_progress_bar=True)

    return chunks, vectors

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
chunks, vectors = embed_chunks(chunks)

print(f"\nVector shape: {vectors.shape}")
print(f"Each chunk becomes a vector of {vectors.shape[1]} numbers")
print(f"\nFirst vector (first 10 numbers): {vectors[0][:10]}")

Embedding 166 chunks...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Vector shape: (166, 384)
Each chunk becomes a vector of 384 numbers

First vector (first 10 numbers): [-0.05015226 -0.03314631 -0.06863559  0.01625643  0.04100114  0.04968648
 -0.00435806  0.00263962  0.11770545 -0.01027864]


The Vector Store (FAISS)

In [ ]:
import faiss
import numpy as np

def build_vector_store(vectors):
    dimension = vectors.shape[1]  # 384

    # L2 = Euclidean distance (standard starting point)
    index = faiss.IndexFlatL2(dimension)

    # FAISS expects float32
    vectors_f32 = np.array(vectors).astype('float32')
    index.add(vectors_f32)

    return index

index = build_vector_store(vectors)
print(f"Vectors in index: {index.ntotal}")

Vectors in index: 166


# Your chunks list IS the lookup table
# FAISS returns index 42 → you do chunks[42] to get the text

# The Retriever

In [ ]:
def retrieve(query: str, index, chunks: list[Document], k: int = 3):
    # 1. Embed the query using the same model
    query_vector = embedder.encode([query]).astype('float32')

    # 2. Search FAISS for the k closest vectors
    distances, indices = index.search(query_vector, k)

    # 3. Use indices to look up the actual chunks
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        chunk.metadata['score'] = float(distances[0][i])
        results.append(chunk)

    return results

In [ ]:
results = retrieve("what is covered for water damage?", index, chunks, k=3)

for i, chunk in enumerate(results):
    print(f"--- Result {i+1} (score: {chunk.metadata['score']:.2f}) ---")
    print(f"Source: {chunk.metadata['filename']}")
    print(f"Text: {chunk.page_content[:300]}")
    print()

--- Result 1 (score: 1.06) ---
Source: Alberta Colleges - 2025 Symposium - FM Presentation (final).pdf
Text: s through domestic water leakage emergency response plans.This is the #1 source of loss at higher education facilities. •Ensuring fire protection systems are always in service and reliable through•Securement of control valves•Impairment management•Regular inspections, testing, and maintenance•Preven

--- Result 2 (score: 1.11) ---
Source: AB Colleges 2025 Symposium - Edmonton October 2025 - Presentation.pdf
Text:  except:
- $100,000 Time Element (Business Interruption) 
- $250,000 Keyano College Properties defined 
- $250,000 Earthquake 
- $2,500 Musical instruments 
- 5% minimum $50,000 subject to maximum $150,000 Water Damage 
- Only provides Property-type insurances 
- Coverage is Property and Equipment B

--- Result 3 (score: 1.26) ---
Source: Alberta Colleges - 2025 Symposium - FM Presentation (final).pdf
Text: elate to the severity and likelihood of property loss at your f

# The Prompt Builder

In [ ]:
def build_prompt(query: str, retrieved_chunks: list[Document]) -> str:
    context = "\n\n---\n\n".join([
        f"Source: {chunk.metadata['filename']}\n{chunk.page_content}"
        for chunk in retrieved_chunks
    ])

    prompt = f"""You are an insurance policy assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer that."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

    return prompt

# Test it
prompt = build_prompt("what is covered for water damage?", results)
print(prompt)

You are an insurance policy assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer that."

CONTEXT:
Source: Alberta Colleges - 2025 Symposium - FM Presentation (final).pdf
s through domestic water leakage emergency response plans.This is the #1 source of loss at higher education facilities. •Ensuring fire protection systems are always in service and reliable through•Securement of control valves•Impairment management•Regular inspections, testing, and maintenance•Prevention of fires through management of ignition sources and hot work•Increasing the percentage of adequately protected buildings by installing automatic sprinklers during renovations and new construction

---

Source: AB Colleges 2025 Symposium - Edmonton October 2025 - Presentation.pdf
 except:
- $100,000 Time Element (Business Interruption) 
- $250,000 Keyano College Properties defined 
- $250,000 Earthquake 
- $2,500 Musical 

In [ ]:
!pip install xai-sdk

In [ ]:
import os
from xai_sdk import Client
from xai_sdk.chat import user
from google.colab import userdata

client = Client(api_key=userdata.get('ragAPIkey'))

def generate_answer(query: str, index, chunks, k: int = 3) -> str:
    # Retrieve
    retrieved = retrieve(query, index, chunks, k)

    # Build prompt
    prompt = build_prompt(query, retrieved)

    # Generate
    chat = client.chat.create(model="grok-4.3")
    chat.append(user(prompt))  # Use user() helper instead of dict
    response = chat.sample()

    return response.content

# Test it
answer = generate_answer("what is covered for water damage?", index, chunks)
print(answer)

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.PERMISSION_DENIED
	details = "Your newly created team doesn't have any credits or licenses yet. You can purchase those on https://console.x.ai/team/4f9b3387-98fc-4f0b-aa25-b5e7d13d87a7."
	debug_error_string = "PERMISSION_DENIED:Your newly created team doesn't have any credits or licenses yet. You can purchase those on https://console.x.ai/team/4f9b3387-98fc-4f0b-aa25-b5e7d13d87a7."
>